## Data Cleansing and Transformation

* Pembersihan data,
* penanganan data yang hilang,
* dan penerapan teknik transformasi untuk mencapai format data yang diinginkan.

In [1]:
# import modul
import pandas as pd

# Transjakarta

In [ ]:
data_trans = pd.read_csv('/content/dfTransjakarta180kRows.csv')
data_trans.info()

FileNotFoundError: [Errno 2] No such file or directory: '/content/dfTransjakarta180kRows.csv'

In [ ]:
kolom_dibuang = [
    'payCardID', 'payCardBank', 'payCardName', 'payCardSex', 'payCardBirthDate', 'tapInStops' , 'payAmount',
    'tapInStopsLat', 'tapInStopsLon',
    'tapOutStops', 'tapOutStopsName', 'tapOutStopsLat', 'tapOutStopsLon',
    'stopEndSeq', 'tapOutTime'
]


df_bersih = data_trans.drop(columns=kolom_dibuang)

df_bersih.isnull().sum()

,0
transID,0
corridorID,6980
corridorName,13528
direction,0
tapInStopsName,0
stopStartSeq,0
tapInTime,0


In [ ]:
# df_bersih.dropna(subset=['corridorID'], inplace=True)
data_referensi = df_bersih.dropna(subset=['corridorName']).drop_duplicates(subset=['corridorID'])
kamus_koridor = dict(zip(data_referensi['corridorID'], data_referensi['corridorName']))
df_bersih['corridorName'] = df_bersih['corridorName'].fillna(df_bersih['corridorID'].map(kamus_koridor))


In [ ]:
df_referensi = df_bersih.dropna(subset=['corridorID']).drop_duplicates(subset=['corridorName'])


kamus_id = dict(zip(df_referensi['corridorName'], df_referensi['corridorID']))

df_bersih['corridorID'] = df_bersih['corridorID'].fillna(df_bersih['corridorName'].map(kamus_id))

In [ ]:
df_bersih['tapInTime'] = pd.to_datetime(df_bersih['tapInTime'])

# 2. Ekstrak bagian tanggalnya saja dan buat kolom baru (misal: 'tanggal_tapIn')
df_bersih['tanggal_tapIn'] = df_bersih['tapInTime'].dt.date
df_bersih["tanggal_tapIn"].value_counts()


,count
tanggal_tapIn,
2023-04-13,10052
2023-04-18,10051
2023-04-19,10050
2023-04-14,10048
2023-04-17,10045
2023-04-28,8065
2023-04-25,8059
2023-04-07,8057
2023-04-12,8052


In [ ]:
df_bersih.head()

,transID,corridorID,corridorName,direction,tapInStopsName,stopStartSeq,tapInTime,tanggal_tapIn
0,VRPJ892P3M98RA,4,Pulo Gadung 2 - Tosari,1.0,Pemuda Rawamangun,12,2023-04-03 06:53:02,2023-04-03
1,ZWCH834I6M26HS,JAK.28,Kp. Rambutan - Taman Wiladatika,1.0,Sekolah Islam PB Soedirman,27,2023-04-03 05:59:19,2023-04-03
2,YRLD835V6L82GO,B13,Bekasi Barat - Blok M,1.0,Mall Metropolitan,6,2023-04-03 05:13:24,2023-04-03
3,ZZBX143N6N83HQ,8K,Batusari - Grogol,1.0,Sbr. Jln. Angsana Kemanggisan,16,2023-04-03 05:20:24,2023-04-03
4,EWEG491A2W45DR,2A,Pulo Gadung - Rawa Buaya via Balai Kota,0.0,BRI Menteng,2,2023-04-03 06:00:54,2023-04-03


# Transjakarta 2

In [12]:
import pandas as pd

# Membaca data
data_angkutan = pd.read_csv('/content/data_angkutan_final_2022_2023.csv')

# Bersihkan kolom jumlah_penumpang_per_hari

# Filter data transjakarta
df_transjakarta = data_angkutan[data_angkutan['jenis_moda'] == 'transjakarta'].copy()

# Ubah tanggal
df_transjakarta['tanggal'] = pd.to_datetime(df_transjakarta['tanggal'], format='mixed')
df_transjakarta['tanggal'] = df_transjakarta['tanggal'].dt.strftime('%Y-%m-%d')
df_transjakarta['jumlah_penumpang_per_hari'] = (
    df_transjakarta['jumlah_penumpang_per_hari']
    .astype(str)
    .str.replace(',', '', regex=False)
    .astype(int)
)
# Urutkan berdasarkan tanggal
df_transjakarta = df_transjakarta.sort_values(by='tanggal')

print(df_transjakarta.head())
print(df_transjakarta.dtypes)

      periode_data     tanggal    jenis_moda  jumlah_penumpang_per_hari
2920        202201  2022-01-01  transjakarta                     450213
2555        202201  2022-01-01  transjakarta                     450213
3285        202201  2022-01-01  transjakarta                     450213
2556        202201  2022-01-02  transjakarta                     367145
2921        202201  2022-01-02  transjakarta                     367145
periode_data                  int64
tanggal                      object
jenis_moda                   object
jumlah_penumpang_per_hari     int64
dtype: object


# api hari libur

In [20]:
import requests
import pandas as pd

import pandas as pd
import requests

url_api = "https://api-hari-libur.vercel.app/api?year=2026"

# Ambil data dari API
response = requests.get(url_api)
data = response.json()

# Ubah bagian 'data' menjadi DataFrame
df = pd.DataFrame(data["data"])

# Tampilkan DataFrame
print(df.head())
data_angkutan.to_csv('hari libur 2026.csv', index=False)

         date                                        description
0  2026-01-01                             Tahun Baru 2026 Masehi
1  2026-01-16                      Isra Mi’raj Nabi Muhammad SAW
2  2026-02-17                     Tahun Baru Imlek 2577 Kongzili
3  2026-03-18  Cuti Bersama Hari Suci Nyepi Tahun Baru Saka 1948
4  2026-03-19               Hari Suci Nyepi Tahun Baru Saka 1948


##Merge data

In [15]:
import pandas as pd
import numpy as np
import requests
import sqlite3

df_transjakarta = pd.read_csv('/content/hasil_merge_baru.csv')
# 1. Standardize both keys to datetime64[ns]
df_transjakarta['tanggal'] = pd.to_datetime(df_transjakarta['tanggal'])
df['date'] = pd.to_datetime(df['date'])

# 2. Create the base 'tipe_hari' column (Weekday vs Weekend)
# .dt.dayofweek returns 0-4 for Mon-Fri, and 5-6 for Sat-Sun
df_transjakarta['tipe_hari'] = np.where(
    df_transjakarta['tanggal'].dt.dayofweek < 5,
    'Weekday',
    'Weekend'
)

# 3. Perform the left join
df_hasil = pd.merge(
    df_transjakarta,
    df,
    left_on='tanggal',
    right_on='date',
    how='left'
)

# 4. Merge holiday descriptions over the default 'tipe_hari'
# If 'description' has a holiday, use it. If it's NaN, keep 'Weekday'/'Weekend'
df_hasil['tipe_hari'] = df_hasil['description'].fillna(df_hasil['tipe_hari'])

# 5. Clean up and reorder
df_hasil = df_hasil.drop(columns=['date', 'description'])
df_hasil.insert(0, 'id', range(1, len(df_hasil) + 1))

# Check the new schema
print(df_hasil.head(10))
df_hasil.to_csv('hasil_merge.csv', index=False)

ValueError: cannot insert id, already exists

In [ ]:
pip install sqlalchemy pymysql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 2.8 MB/s eta 0:00:00


KeyboardInterrupt: 

In [ ]:
from sqlalchemy import create_engine



engine = create_engine(
    DATABASE_URL,
    connect_args={
        "ssl": {
            "ssl_mode": "REQUIRED"
        }
    }
)
df_hasil.to_sql(
    "transjakarta_harian",
    engine,
    if_exists="append",
    index=False,
    chunksize=1000,
    method="multi"
)


365

In [ ]:
from sqlalchemy import create_engine, text

DATABASE_URL = (
    "mysql+pymysql://avnadmin:AVNS_VjGrndn1iv-tJ3l0oTH"
    "@mysql-bad9f57-anggiwahyu538-76aa.l.aivencloud.com:26274/transjakarta"
)

engine = create_engine(
    DATABASE_URL,
    connect_args={
        "ssl": {
            "ssl_mode": "REQUIRED"
        }
    }
)

# Buat tabel
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS transjakarta_harian (
            id INT PRIMARY KEY,
            periode_data INT NOT NULL,
            tanggal DATE NOT NULL,
            jenis_moda VARCHAR(50) NOT NULL,
            jumlah_penumpang_per_hari INT NOT NULL,
            tipe_hari VARCHAR(100) NOT NULL
        );
    """))

print("Tabel berhasil dibuat")

Tabel berhasil dibuat
